# Week 2, Lab 4: RAG 101 - Your Digital Twin Finally Gets Its Memory

## The map before the territory

A model only knows what it was trained on. It has never heard of Ada's bakery or her marathon plans. **RAG** stands for *Retrieval-Augmented Generation*. It fixes this in three steps:

1. **Retrieve**: find the few chunks of your own text that best match the question. You do this with embeddings.
2. **Augment**: paste those chunks into the prompt, next to the question.
3. **Generate**: the model answers, using the chunks you gave it.

The place where the chunk vectors live, and where the search happens, is a **vector database**. Real ones (Chroma, FAISS, pgvector) are built for millions of vectors. This lab builds a tiny one yourself so you see exactly what it does. There is no API call: you stop at step 2 and print the prompt that a real call would send. The Reading tab explains classes that hold a list, `sorted` with a key, and building a prompt from a list from scratch if you want a refresher first.

## Exercise

### Goal
Build the heart of a tiny vector database: a `VectorStore` that holds texts together with their vectors and can find the best matches for a new vector. `__init__` and `add` are already written, and so is `cosine_similarity` from lab 3. You write `search`.

### 1. `VectorStore.search(self, query_vector, k)`
Code shape (replace each `...`):
```python
def search(self, query_vector, k):
    scored = [(..., text) for text, vector in self.items]
    scored.sort(...)
    return scored[:k]
```
A *class* bundles data and functions together. A *method* is a function inside a class. `self` is the object itself. In `__init__`, `self.items = []` makes an empty list. Each call to `add(text, vector)` puts one `(text, vector)` pair at the end of it.

**You get:** `query_vector`, a list of numbers, and `k`, how many results you want.
**You give back:** a list of at most `k` pairs `(score, text)`, with the best match first. `score` is the cosine similarity between `query_vector` and that text's vector.

**Idea:**
1. Go over `self.items`. For each `(text, vector)` make a pair `(cosine_similarity(query_vector, vector), text)`. A list comprehension does it in one line.
2. Sort the pairs so the highest score comes first. `scored.sort(reverse=True)` sorts pairs by their first item, the score, and `reverse=True` puts the biggest first.
3. Keep only the first `k`: `scored[:k]` is a slice with at most `k` items.

- If you ask for more results than the store holds, you get all of them. A slice does not complain about this.
- An empty store gives `[]`.
- The store does not change when you search.

**Check (by hand):** a store with `"a"` at `[1, 0]`, `"b"` at `[0, 1]` and `"c"` at `[1, 1]`. Searching with `[1, 0]` and `k=2` scores `a` as 1.0, `c` as about 0.707 and `b` as 0.0. You get `[(1.0, "a"), (0.707..., "c")]`.

### What the asserts below test
The cell below the class builds that three-item store and checks four things:
- `search([1, 0], 2)` gives two results;
- the texts come back in the order `"a"`, then `"c"`;
- the first score is `1.0` and the second is `0.707` (rounded to 3 decimal places);
- `search([1, 0], 10)` gives all three items, and an empty store gives `[]`.

If all four hold, you see `All checks passed.`.

In [ ]:
import math


def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    length_a = math.sqrt(sum(x * x for x in a))
    length_b = math.sqrt(sum(x * x for x in b))
    if length_a == 0 or length_b == 0:
        return 0.0
    return dot / (length_a * length_b)


class VectorStore:
    def __init__(self):
        self.items = []

    def add(self, text: str, vector: list[float]) -> None:
        self.items.append((text, vector))

    def search(self, query_vector: list[float], k: int) -> list[tuple[float, str]]:
        """Find the k texts whose vectors are closest to query_vector.

        You get: a query vector (list of numbers) and k, how many results to give back.
        You give back: a list of at most k pairs (score, text), best match first,
        where score is the cosine similarity between query_vector and the text's vector.
        """
        # TODO:
        # 1. scored = a list of (cosine_similarity(query_vector, vector), text) for each (text, vector) in self.items.
        # 2. Sort scored with scored.sort(reverse=True) so the best score is first.
        # 3. Give back the first k items: scored[:k].
        raise NotImplementedError("VectorStore.search() is not implemented yet - replace the TODO with your code")


store = VectorStore()
store.add("a", [1, 0])
store.add("b", [0, 1])
store.add("c", [1, 1])

top = store.search([1, 0], 2)
assert len(top) == 2, "search([1, 0], 2) should give 2 results, got " + repr(top)
assert [text for _, text in top] == ["a", "c"], 'the texts should come back as ["a", "c"], got ' + repr([text for _, text in top])
assert round(top[0][0], 3) == 1.0 and round(top[1][0], 3) == 0.707, "the scores should be 1.0 and 0.707, got " + repr([round(score, 3) for score, _ in top])
assert len(store.search([1, 0], 10)) == 3, "asking for more than the store holds should give all 3 items"
assert VectorStore().search([1, 0], 2) == [], "an empty store should give []"

print("All checks passed.")


## Solution

In [ ]:
class VectorStore:
    def __init__(self):
        self.items = []

    def add(self, text: str, vector: list[float]) -> None:
        self.items.append((text, vector))

    def search(self, query_vector: list[float], k: int) -> list[tuple[float, str]]:
        scored = [(cosine_similarity(query_vector, vector), text) for text, vector in self.items]
        scored.sort(reverse=True)
        return scored[:k]


store = VectorStore()
store.add("a", [1, 0])
store.add("b", [0, 1])
store.add("c", [1, 1])
print(store.search([1, 0], 2))


## Real-world project (offline)

The code cell below loads the six facts about Ada, the tiny word vectors from lab 3 and an `embed` function that turns a sentence into a vector. Your project fills a `VectorStore` with the facts, then answers two questions the RAG way: retrieve the best facts, paste them into a prompt and print it. Nothing here calls an LLM. The printed prompt is exactly what the week 1 `GroqClient` would send.

In [ ]:
"""Load the twin's facts and the toy word vectors (embedded so this notebook needs no repo checkout)."""
# lab-data-loader
# Each word gets 5 numbers: how much it means food, tech, sport, travel, school.
WORD_VECTORS = {
    # food
    "food": [1.0, 0.0, 0.0, 0.0, 0.0],
    "cook": [0.9, 0.0, 0.0, 0.0, 0.0],
    "cooks": [0.9, 0.0, 0.0, 0.0, 0.0],
    "bakes": [0.9, 0.0, 0.0, 0.0, 0.0],
    "bread": [1.0, 0.0, 0.0, 0.0, 0.0],
    "soup": [1.0, 0.0, 0.0, 0.0, 0.0],
    "spicy": [0.8, 0.0, 0.0, 0.1, 0.0],
    "eat": [0.9, 0.0, 0.1, 0.0, 0.0],
    # tech
    "python": [0.0, 1.0, 0.0, 0.0, 0.0],
    "code": [0.0, 0.9, 0.0, 0.0, 0.0],
    "builds": [0.0, 0.7, 0.0, 0.0, 0.0],
    "web": [0.0, 0.9, 0.0, 0.1, 0.0],
    "apps": [0.0, 1.0, 0.0, 0.0, 0.0],
    "programming": [0.0, 1.0, 0.0, 0.0, 0.0],
    "language": [0.0, 0.6, 0.0, 0.1, 0.0],
    "data": [0.0, 0.7, 0.0, 0.0, 0.3],
    "machine": [0.0, 0.7, 0.0, 0.0, 0.3],
    "learning": [0.0, 0.4, 0.0, 0.0, 0.6],
    # sport
    "runs": [0.0, 0.0, 1.0, 0.0, 0.0],
    "run": [0.0, 0.0, 1.0, 0.0, 0.0],
    "kilometers": [0.0, 0.0, 0.8, 0.2, 0.0],
    "park": [0.0, 0.0, 0.6, 0.2, 0.0],
    "team": [0.0, 0.0, 0.9, 0.0, 0.0],
    "match": [0.0, 0.0, 1.0, 0.0, 0.0],
    "watches": [0.0, 0.1, 0.4, 0.0, 0.0],
    "sport": [0.0, 0.0, 1.0, 0.0, 0.0],
    "football": [0.0, 0.0, 1.0, 0.0, 0.0],
    "exercise": [0.0, 0.0, 1.0, 0.0, 0.0],
    # travel
    "travelled": [0.0, 0.0, 0.0, 1.0, 0.0],
    "plane": [0.0, 0.0, 0.0, 1.0, 0.0],
    "accra": [0.0, 0.0, 0.0, 0.9, 0.0],
    "lisbon": [0.0, 0.0, 0.0, 0.9, 0.0],
    "summer": [0.0, 0.0, 0.0, 0.5, 0.0],
    "countries": [0.0, 0.0, 0.0, 1.0, 0.0],
    "visit": [0.0, 0.0, 0.0, 1.0, 0.0],
    "trip": [0.0, 0.0, 0.0, 1.0, 0.0],
    # school
    "teaches": [0.0, 0.0, 0.0, 0.0, 1.0],
    "teach": [0.0, 0.0, 0.0, 0.0, 1.0],
    "classes": [0.0, 0.0, 0.0, 0.0, 1.0],
}

TWIN_FACTS = [
    "Ada writes Python code and builds web apps every day.",
    "She bakes bread and cooks spicy soup on weekends.",
    "On Saturdays she runs five kilometers in the park.",
    "Last summer she travelled to Accra and Lisbon by plane.",
    "She teaches evening classes about data and machine learning.",
    "Her favorite team is Hearts of Oak and she watches every match.",
]


def embed(text):
    words = text.lower().replace(".", "").replace("?", "").split()
    known = [WORD_VECTORS[w] for w in words if w in WORD_VECTORS]
    return [sum(column) / len(known) for column in zip(*known)]

print(f"Loaded {len(TWIN_FACTS)} facts and {len(WORD_VECTORS)} word vectors.")


In [ ]:
store = VectorStore()
for fact in TWIN_FACTS:
    store.add(fact, embed(fact))

questions = [
    "Which sport team does she follow?",
    "Where has she travelled?",
]

for question in questions:
    results = store.search(embed(question), 2)
    context = "\n".join(f"- {text}" for _, text in results)
    prompt = f"Answer using only this context.\n\nContext:\n{context}\n\nQuestion: {question}"
    print(prompt)
    print("=" * 20)
